In [ ]:
!pip install torch

In [ ]:
import os, math, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
#1. Lets first do the possible config
preset="gpu" if torch.cuda.is_available() else "cpu"
presets={
    "cpu":dict(block_size=64,batch_size=32,n_layer=4,n_head=4,n_embd=128,max_iters=3000,lr=1e-3),
    "gpu":dict(block_size=256,batch_size=64,n_layer=6,n_head=6,n_embd=384,max_iters=5000,lr=6e-4),
}
cfg=presets[preset]
cfg["max_iters"]=int(os.environ.get("MAX_ITERS",cfg["max_iters"])) #override for quick tests
dropout=0.1
eval_interval,eval_iters=250,50
warmup_iters=100

device="cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
torch.manual_seed(1337)
# 2. DATA  (read text -> build vocabulary -> convert to numbers -> split)
DATA_PATH = "data/input.txt"
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
os.makedirs("data", exist_ok=True)
if not os.path.exists(DATA_PATH):
    print("Downloading TinyShakespeare...")
    urllib.request.urlretrieve(URL, DATA_PATH)

text = open(DATA_PATH, encoding="utf-8").read()
chars = sorted(set(text))                       # every unique character = our vocabulary
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}      # character -> number
itos = {i: c for c, i in stoi.items()}          # number -> character
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))                        # 90% train, 10% validation
train_data, val_data = data[:n], data[n:]
print(f"{len(text):,} characters | vocab size {vocab_size} | device {device} | preset {preset}")

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - cfg["block_size"] - 1, (cfg["batch_size"],))
    x = torch.stack([d[i : i + cfg["block_size"]] for i in ix])          # input tokens
    y = torch.stack([d[i + 1 : i + cfg["block_size"] + 1] for i in ix])  # same, shifted by 1 = the answers
    return x.to(device), y.to(device)

1,115,394 characters | vocab size 65 | device cuda | preset gpu


In [ ]:
# 3. MODEL  (exactly the parts you learned)
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)   # makes Q, K, V together
        self.proj = nn.Linear(n_embd, n_embd)      # this is W_O
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):                          # x: (B, T, C)
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # split into heads: (B, T, C) -> (B, heads, T, C/heads)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # softmax(QK^T / sqrt(d)) V with the causal mask, done efficiently by PyTorch
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)   # glue heads back together
        return self.resid_drop(self.proj(y))

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = nn.Sequential(                  # expand -> activate -> shrink
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))             # communication  (+ residual)
        x = x + self.ffn(self.ln2(x))              # computation    (+ residual)
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)    # token -> vector
        self.pos_emb = nn.Embedding(block_size, n_embd)    # position -> vector (learned)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)  # vector -> score per token
        self.head.weight = self.tok_emb.weight                 # weight tying: saves parameters
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.blocks(self.drop(x))
        logits = self.head(self.ln_f(x))                       # (B, T, vocab)
        loss = None
        if targets is not None:                                # cross-entropy = how wrong the predictions are
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])        # only the last block_size tokens fit
            logits = logits[:, -1, :] / temperature            # scores for the NEXT token
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")    # keep only top-k options
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)      # sample one token
            idx = torch.cat([idx, nxt], dim=1)                 # append and repeat
        return idx

model = GPT(vocab_size, cfg["block_size"], cfg["n_layer"], cfg["n_head"], cfg["n_embd"]).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

Model parameters: 10.77M


In [ ]:
# 4. TRAINING  (predict next token -> measure loss -> nudge weights -> rep
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.1)

def get_lr(it):                                    # warmup, then cosine decay
    if it < warmup_iters:
        return cfg["lr"] * (it + 1) / warmup_iters
    progress = (it - warmup_iters) / max(1, cfg["max_iters"] - warmup_iters)
    return cfg["lr"] * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress)))

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            losses[k] = model(xb, yb)[1].item()
        out[split] = losses.mean().item()
    model.train()
    return out

print(f"Random-guess loss would be about {math.log(vocab_size):.2f}")
start = time.time()
for it in range(cfg["max_iters"] + 1):
    for g in optimizer.param_groups:
        g["lr"] = get_lr(it)
    if it % eval_interval == 0 or it == cfg["max_iters"]:
        l = estimate_loss()
        print(f"step {it:5d} | train loss {l['train']:.3f} | val loss {l['val']:.3f} | {time.time()-start:.0f}s")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # stops exploding gradients
    optimizer.step()


Random-guess loss would be about 4.17
step     0 | train loss 4.245 | val loss 4.239 | 12s
step   250 | train loss 2.019 | val loss 2.101 | 115s
step   500 | train loss 1.519 | val loss 1.699 | 225s
step   750 | train loss 1.351 | val loss 1.575 | 340s
step  1000 | train loss 1.250 | val loss 1.518 | 456s
step  1250 | train loss 1.175 | val loss 1.481 | 573s
step  1500 | train loss 1.114 | val loss 1.463 | 688s
step  1750 | train loss 1.042 | val loss 1.482 | 805s
step  2000 | train loss 0.978 | val loss 1.513 | 922s
step  2250 | train loss 0.909 | val loss 1.542 | 1038s
step  2500 | train loss 0.832 | val loss 1.587 | 1154s
step  2750 | train loss 0.757 | val loss 1.615 | 1270s
step  3000 | train loss 0.678 | val loss 1.691 | 1386s
step  3250 | train loss 0.611 | val loss 1.753 | 1502s
step  3500 | train loss 0.549 | val loss 1.790 | 1619s
step  3750 | train loss 0.495 | val loss 1.851 | 1735s
step  4000 | train loss 0.450 | val loss 1.900 | 1851s
step  4250 | train loss 0.411 | val l

In [ ]:
# 5. SAVE + GENERATE

torch.save({"model": model.state_dict(), "cfg": cfg, "chars": chars}, "mini_gpt.pt")
print("Saved to mini_gpt.pt\n--- generated text ---")
model.eval()
start_ids = torch.zeros((1, 1), dtype=torch.long, device=device)   # start from token 0
print(decode(model.generate(start_ids, max_new_tokens=500)[0].tolist()))

Saved to mini_gpt.pt
--- generated text ---

But now the king shall forbear the lives.

KING RICHARD III:
So:
What am I sabel, I come hither, sir?

DUKE OF AUMERLE:
Comfort, my liege; remember who you are.

KING RICHARD II:
Ay, but thou wilt say to me, I say.

DUCHESS OF YORK:
Why, brother, I cannot tell it so,
I am content: but 'tis thy wisdom to come;
But, that thou shalt not, thy heavy sound companion
That is thy old cause on thee, which never
Shall know more willingly come to come up both.
I know thy dukedom, and tell me what says
My h
